# BÁO CÁO THỰC HÀNH 2
## Transfer Learning với VGG16 và DenseNet121

**Học phần:** Mạng nơ-ron tích chập và học chuyển giao  
**Bài toán:** Phân loại 37 giống chó/mèo trên Oxford-IIIT Pet  
**Nhóm:** Nhóm 10  

---

### Tóm tắt

Báo cáo trình bày quy trình từ chuẩn bị dữ liệu, tiền xử lý ảnh, xây dựng mô hình học chuyển giao, huấn luyện và chọn checkpoint, đến đánh giá kết quả. Nhóm sử dụng VGG16 và DenseNet121 với trọng số pretrained ImageNet. Trong các cấu hình đã chạy, **DenseNet121 với learning rate 0.001 và batch size 16 đạt test accuracy 91.31%**, cao hơn VGG16 2.51 điểm phần trăm trong lần thực nghiệm này.

Notebook này vừa là tài liệu báo cáo, vừa có các ô mã để kiểm tra dữ liệu, nạp một batch và đọc lại kết quả đã lưu. Việc huấn luyện đầy đủ được để ở một ô lệnh riêng vì có thể mất nhiều thời gian.

## 1. Tên đề tài

### Phân loại giống chó, mèo bằng học chuyển giao

| Bài toán | Dữ liệu | Mô hình |
|---|---|---|
| Phân loại 37 giống chó/mèo | Oxford-IIIT Pet | VGG16 · DenseNet121 |

## 2. Hướng chính

**Thị giác máy tính · Transfer Learning**

Dùng VGG16 và DenseNet121 pretrained trên ImageNet, thay lớp phân loại cuối thành 37 lớp và so sánh kết quả trên cùng bộ dữ liệu.

> Phạm vi: thực nghiệm và đánh giá mô hình; chưa triển khai ứng dụng thời gian thực.

## 3. Người sử dụng và vấn đề

| Đối tượng | Nhu cầu |
|---|---|---|
| Sinh viên, giảng viên | Học và đánh giá mô hình phân loại ảnh. |
| Người xem demo | Thử dự đoán giống chó/mèo từ ảnh. |

**Vấn đề:** phân biệt nhiều giống bằng mắt có thể khó; mô hình được khảo sát như công cụ tham khảo, không thay thế đánh giá của con người.

## 4. Mục tiêu

Xây dựng bộ phân loại 37 lớp và so sánh VGG16 với DenseNet121 trên Oxford-IIIT Pet.

| Nội dung đánh giá | Chỉ số / thiết lập |
|---|---|
| So sánh mô hình | Validation loss, accuracy; test cho cấu hình được chọn |
| Learning rate | 0.01 và 0.001 |
| Batch size | 16 (cố định trong kết quả hiện có) |

*Kết quả hiện tại dựa trên một seed và một split.*

## 5. Dữ liệu Oxford-IIIT Pet

Bộ dữ liệu có 37 lớp, được tổ chức theo cấu trúc `ImageFolder`. Project nhận sẵn ba thư mục `train`, `val`, `test`; mã nguồn kiểm tra các thư mục, ảnh và tên lớp nhưng không tự tạo hoặc chia lại split.

| Tập | Số ảnh | Vai trò |
|---|---:|---|
| Train | 2,941 | Cập nhật trọng số mô hình |
| Validation | 739 | Đánh giá sau mỗi epoch; chọn checkpoint/cấu hình theo validation loss thấp nhất |
| Test | 3,669 | Đánh giá cấu hình được chọn bằng validation |

![Một số ảnh thuộc Oxford-IIIT Pet](../bao_cao_hinh_anh/anh_mau_oxford_pet.png)

Hình minh họa các giống khác nhau trong bài toán. Tên thư mục lớp được dùng làm nhãn; cả ba split phải có cùng tên lớp và cùng thứ tự lớp để chỉ số nhãn nhất quán.

Nguồn chính thức: [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/). Trang dataset ghi giấy phép [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) và bản quyền ảnh thuộc chủ sở hữu gốc. Project chưa ghi nhận bộ dữ liệu thay thế.

### Phân bố dữ liệu theo split

![Số lượng ảnh trong từng split](../bao_cao_hinh_anh/phan_bo_dataset.png)

Mỗi epoch, chương trình đo loss và accuracy trên validation; checkpoint được lưu khi validation loss giảm. Sau các lần chạy, mỗi kiến trúc chọn cấu hình có validation loss thấp nhất; test chỉ đánh giá cấu hình đã chọn. Validation không cập nhật trọng số.

Theo thông tin chuẩn bị dữ liệu được ghi nhận, train/validation được tách phân tầng 80/20 từ trainval và test dùng split chính thức. Tuy nhiên, mã nguồn hiện tại chỉ nạp các thư mục split có sẵn, không có bước tạo split để kiểm chứng độc lập. Kiểm tra checksum SHA-256 không phát hiện ảnh byte-identical giữa các split; cách này không phát hiện ảnh đã nén lại hoặc ảnh gần giống.

In [ ]:
# Tìm thư mục gốc của dự án dù notebook được mở từ thư mục notebooks/ hay từ repo.
from pathlib import Path

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "main.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy src/main.py; hãy mở notebook trong repo BAI_E2.")

DATA_DIR = ROOT / "dataset"
print(f"Thư mục dự án: {ROOT}")
print(f"Thư mục dữ liệu: {DATA_DIR}")

In [ ]:
# Kiểm tra nhanh số lớp, số ảnh và tính nhất quán tên lớp giữa các split.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
split_classes = {}
split_counts = {}

for split in ("train", "val", "test"):
    split_dir = DATA_DIR / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Thiếu thư mục dữ liệu: {split_dir}")
    classes = sorted(path.name for path in split_dir.iterdir() if path.is_dir())
    split_classes[split] = classes
    split_counts[split] = sum(
        1 for path in split_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    print(f"{split:>5}: {split_counts[split]:>5} ảnh | {len(classes):>2} lớp")

if not (split_classes["train"] == split_classes["val"] == split_classes["test"]):
    raise ValueError("Tên lớp giữa train, val và test không đồng nhất.")
print("Các split có cùng danh sách lớp:", len(split_classes["train"]) == 37)

## 6. Phạm vi

| Đã thực hiện | Chưa thực hiện |
|---|---|
| VGG16, DenseNet121; LR 0.01/0.001; batch 16 | Batch 32; nhiều seed/k-fold; ứng dụng/API |

Cấu hình được chọn bằng validation; test chỉ chạy sau khi chọn.

## 7. Phương pháp

| Bước | Cách thực hiện |
|---|---|
| Tiền xử lý | Resize/crop 224×224, chuẩn hóa theo ImageNet |
| Train | Random crop + lật ngang |
| Validation / test | Center crop, không augmentation |
| Mô hình | VGG16 hoặc DenseNet121 pretrained; classifier 37 lớp |
| Huấn luyện | CrossEntropyLoss + Adam; chọn checkpoint theo validation loss |

Chi tiết trong [`src/data_loader.py`](../src/data_loader.py), [`src/models.py`](../src/models.py) và [`src/engine.py`](../src/engine.py).

In [ ]:
# Dùng chính DataLoader của dự án để xác nhận một batch có thể được tạo.
import sys

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from data_loader import create_dataloaders

train_loader, train_eval_loader, val_loader, test_loader, class_names = create_dataloaders(
    DATA_DIR, batch_size=16, num_workers=0, seed=42
)
images, labels = next(iter(train_loader))
print("Số lớp:", len(class_names))
print("Shape ảnh trong batch:", tuple(images.shape))
print("Shape nhãn:", tuple(labels.shape))
print("Một vài nhãn:", [class_names[index] for index in labels[:5].tolist()])

Batch đầu vào: `[16, 3, 224, 224]`.

### Mô hình và học chuyển giao

Cả hai mô hình dùng trọng số ImageNet, classifier 37 lớp và đóng băng backbone. Kết quả hiện tại không bật fine-tuning.

### Sơ đồ quá trình

![Sơ đồ quy trình thực nghiệm từ dữ liệu đến đánh giá](../bao_cao_hinh_anh/so_do_quy_trinh.svg)

### Huấn luyện, validation và chọn mô hình

| Giai đoạn | Quy trình |
|---|---|
| Train | CrossEntropyLoss → Adam cập nhật classifier |
| Validation | Ghi loss/accuracy, lưu checkpoint loss thấp nhất |
| Dừng sớm | Tối đa 5 epoch; patience 3 |
| Test | Chỉ đánh giá cấu hình được chọn bằng validation |

Mã ghi metric vào TensorBoard; event log được tạo khi chạy thí nghiệm. CSV tổng hợp nằm tại `outputs/reduced-experiments/results.csv`.

In [ ]:
# Lệnh tái chạy ma trận thí nghiệm đầy đủ (tối đa 8 lần huấn luyện).
# Bỏ dấu # và chạy riêng ô này khi đã cài requirements và sẵn sàng chờ huấn luyện.
# !python src/main.py --data-dir dataset --epochs 5 --patience 3 --batch-size 16 --tune --output-dir outputs/reduced-experiments

Kết quả hiện có: 4 cấu hình, seed 42, batch size 16. Batch size 32 chưa được chạy.

Sau khi chạy thí nghiệm để tạo event log, mở TensorBoard:

```powershell
python -m tensorboard.main --logdir .\outputs\reduced-experiments
```

## 8. Sản phẩm

| Thành phần | Thư mục |
|---|---|
| Mã nguồn mô hình/huấn luyện | `src/` |
| Báo cáo | `notebooks/` |
| Kết quả CSV | `outputs/reduced-experiments/results.csv` |
| TensorBoard | `src/tensorboard_logger.py` — ghi loss/accuracy; event log được tạo khi chạy thí nghiệm |
| Hình và biểu đồ | `bao_cao_hinh_anh/` |

**Lưu ý:** hiện `outputs/` có CSV nhưng chưa có event log TensorBoard. Chưa có ứng dụng triển khai hoặc API.

## 9. Đánh giá

Kết quả được nạp trực tiếp từ `results.csv`. Test chỉ có ở cấu hình được chọn bằng validation.

In [ ]:
import csv

RESULTS_PATH = ROOT / "outputs" / "reduced-experiments" / "results.csv"
if not RESULTS_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy file kết quả: {RESULTS_PATH}")

with RESULTS_PATH.open("r", encoding="utf-8-sig", newline="") as results_file:
    results = list(csv.DictReader(results_file))

columns = ("Model", "LR", "Batch", "Epoch tốt nhất", "Val loss", "Val acc", "Test acc", "Chọn theo val")
print(" | ".join(columns))
print(" | ".join("---" for _ in columns))
for row in results:
    test_accuracy = f"{float(row['test_accuracy']) * 100:.2f}%" if row["test_accuracy"] else "—"
    selected = "Có" if row["selected_by_validation"].lower() == "true" else "Không"
    values = (
        row["model"], row["learning_rate"], row["batch_size"], row["best_epoch"],
        f"{float(row['best_validation_loss']):.4f}",
        f"{float(row['validation_accuracy_at_best_loss']) * 100:.2f}%",
        test_accuracy, selected,
    )
    print(" | ".join(values))

### So sánh kết quả

Xanh lá: cấu hình được chọn theo validation. Test chỉ hiển thị cấu hình đã chọn.

In [ ]:
import matplotlib.pyplot as plt

labels = [f"{row['model']}\nLR={float(row['learning_rate']):g}" for row in results]
selected_flags = [row["selected_by_validation"].lower() == "true" for row in results]
bar_colors = ["#2e8b57" if selected else "#9aa7b2" for selected in selected_flags]
validation_accuracies = [float(row["validation_accuracy_at_best_loss"]) * 100 for row in results]
validation_losses = [float(row["best_validation_loss"]) for row in results]
selected_results = [row for row in results if row["test_accuracy"]]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
fig.suptitle("So sánh cấu hình theo validation và test", fontsize=16, fontweight="bold")

bars = axes[0].bar(labels, validation_accuracies, color=bar_colors)
axes[0].set_title("Validation accuracy ↑")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(0, 105)
axes[0].bar_label(bars, fmt="%.1f%%", padding=3)

bars = axes[1].bar(labels, validation_losses, color=bar_colors)
axes[1].set_title("Validation loss ↓")
axes[1].set_ylabel("Cross-entropy loss")
axes[1].bar_label(bars, fmt="%.3f", padding=3)

test_labels = [row["model"] for row in selected_results]
test_accuracies = [float(row["test_accuracy"]) * 100 for row in selected_results]
bars = axes[2].bar(test_labels, test_accuracies, color="#3977b8")
axes[2].set_title("Test accuracy (đã chọn) ↑")
axes[2].set_ylabel("Accuracy (%)")
axes[2].set_ylim(0, 105)
axes[2].bar_label(bars, fmt="%.1f%%", padding=3)

for axis in axes:
    axis.grid(axis="y", alpha=0.2)
    axis.set_axisbelow(True)
    axis.tick_params(axis="x", labelrotation=0, labelsize=9)
fig.tight_layout()
plt.show()

Cả hai mô hình chọn LR 0.001 theo validation loss. DenseNet121 đạt test accuracy cao hơn 2.51 điểm phần trăm trong lần chạy này.

### Tóm tắt kết quả được chọn

| Kiến trúc | Learning rate | Batch size | Validation loss | Validation accuracy | Test loss | Test accuracy |
|---|---:|---:|---:|---:|---:|---:|
| VGG16 | 0.001 | 16 | 0.2831 | 91.34% | 0.3450 | 88.80% |
| DenseNet121 | 0.001 | 16 | 0.2767 | 91.88% | 0.3236 | 91.31% |

Kết quả được chọn theo validation; cấu hình không được chọn không chạy test.

### Kết luận

**DenseNet121** đạt kết quả tốt hơn trong lần chạy hiện tại. Kết luận chỉ áp dụng cho split và cấu hình đã thử.

## 10. Kế hoạch và rủi ro

Các bước chính đã hoàn thành; project chưa ghi lịch ngày cụ thể. Rủi ro chính là thời gian chạy và kết quả thay đổi theo split/seed. Nhóm dùng early stopping và chỉ đánh giá test sau khi chọn cấu hình bằng validation.

Bảng phân công để nhóm cập nhật:

| Thành viên | Nhiệm vụ | Mức độ hoàn thành |
| --- | --- | --- |
|  |  |  |

### Hạn chế và hướng cải thiện

| Hạn chế hiện tại | Có thể làm tiếp |
|---|---|
| Batch size chỉ có 16 | Thử thêm batch size 32 |
| Một seed, một split; tối đa 5 epoch | Chạy thêm seed hoặc tăng số epoch |
| Chưa phân tích nhầm lẫn theo lớp | Thêm confusion matrix |
| Kiểm tra trùng ảnh bằng SHA-256 | Kiểm tra thêm ảnh gần giống |

### Kết luận báo cáo

Đã xây dựng và đánh giá mô hình phân loại 37 giống chó/mèo bằng VGG16 và DenseNet121. Trong lần chạy hiện tại, DenseNet121 đạt test accuracy **91.31%**, so với **88.80%** của VGG16. Kết quả chỉ phản ánh split và cấu hình đã thử.

### Tài liệu/mã nguồn liên quan

- [`README.md`](../README.md) — hướng dẫn tổng quan và cách chạy dự án.
- [`src/data_loader.py`](../src/data_loader.py) — tiền xử lý và DataLoader.
- [`src/models.py`](../src/models.py) — kiến trúc VGG16/DenseNet121.
- [`src/engine.py`](../src/engine.py) — huấn luyện và đánh giá theo epoch.
- [`src/main.py`](../src/main.py) — điều phối thí nghiệm, early stopping và lưu kết quả.
- [`src/tensorboard_logger.py`](../src/tensorboard_logger.py) — ghi metric và hyperparameter vào TensorBoard.
- [`docs/bao_cao_kiem_tra_du_lieu_va_mo_hinh.md`](../docs/bao_cao_kiem_tra_du_lieu_va_mo_hinh.md) — chi tiết kiểm tra dữ liệu và mô hình.